# Thinking about flows

Adapted from the [Monash EMU summer textbook](https://github.com/monash-emu/summer-textbook)
notebook `textbook/03-flows-introduction.ipynb` at commit
`fd97783474789e50ace5ea420aec20147f9bbd76`.

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

Taking a step back from the simple model in the previous chapter, this notebook
digs into what we mean by a **transition flow**. We work through some simple
maths not because the maths is interesting on its own, but to cement the
intuition of what a "flow" is. To do this, consider a really simple model
without an infection process — only a single transition between two
compartments.

![](figures/03/source_dest_structure.svg)


In [ ]:
from typing import Any

import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    Compartments,
    FlowModel,
    Param,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

AXIS = {"index": "time", "value": "proportion"}


def state_frame(trace: Any) -> pd.DataFrame:
    """Compartment trace as a DataFrame with short state labels."""
    frame = trace.to_pandas()
    frame.columns = [c.split("state=")[-1].split("_")[0] for c in frame.columns]
    return frame


In [ ]:
def get_single_transition_model() -> Any:
    """Two compartments linked by one per-capita transition flow."""
    state = Property("state", ("source", "destination"))
    pmap = PropertyMap.from_property(state)
    model = FlowModel(pmap)
    model.add_flow(
        TransitionFlow(
            "transition",
            state["source"],
            state["destination"],
            Param("transition_rate"),
        )
    )
    return model.compile(), pmap, state


model_config = {"population": 1.0, "end_time": 20.0}
parameters = {"transition_rate": 0.1}

cm, pmap, state = get_single_transition_model()
y0 = np.zeros(pmap.size)
y0[pmap.select(state["source"])] = model_config["population"]
times = np.linspace(0.0, model_config["end_time"], int(model_config["end_time"]) + 1)
plan = SavePlan(requests={"comp": SaveRequest(Compartments())}, ts=times)
res = cm.run(parameters, y0, t0=0.0, t1=model_config["end_time"], dt=1.0, save=plan, solver="dopri5")
model_df = state_frame(res["comp"])
model_df.plot(labels=AXIS, title="Single transition flow")


With this single transition flow, people in the source compartment move to the
destination compartment at a *per capita* rate given by the flow parameter.
By "per capita" we mean the absolute flow rate is the product of that parameter
and the size of the source compartment — summer4 multiplies the declared rate
by the source population automatically for a non-`absolute` `TransitionFlow`.


## Analytic equivalent

For a model this simple, the same system can be solved analytically. As we add
compartments and complexity we quickly reach the point where we cannot predict
sizes without solving the system numerically (see
[obtaining numerical solutions](./07-numerical-solutions.ipynb)).

If $S(t)$ and $D(t)$ are the source and destination sizes,

$$
\frac{dS(t)}{dt} = -\mathrm{rate}\, S(t), \qquad
\frac{dD(t)}{dt} = \mathrm{rate}\, S(t).
$$

With $D(0)=0$, the closed form is $S(t)=S(0)\,e^{-\mathrm{rate}\,t}$ and
$D(t)=S(0)\,(1-e^{-\mathrm{rate}\,t})$. Check that the numerical run matches.


In [ ]:
rate = parameters["transition_rate"]
analytic = pd.DataFrame(
    {
        "source": np.exp(-rate * times),
        "destination": 1.0 - np.exp(-rate * times),
    },
    index=times,
)
analytic.plot(labels=AXIS, title="Analytic exponential decay")

np.testing.assert_allclose(model_df["source"].to_numpy(), analytic["source"].to_numpy(), rtol=1e-4, atol=1e-4)
np.testing.assert_allclose(
    model_df["destination"].to_numpy(), analytic["destination"].to_numpy(), rtol=1e-4, atol=1e-4
)
assert float(model_df["source"].iloc[-1]) < float(model_df["source"].iloc[0])


## Other flow types

In infectious-disease modelling, transition flows represent progression from
one state to another (for example recovery after infection). They are not the
only flow type summer4 supports. The short table below is the summer4 shape of
the same idea; infection flows use `ForceOfInfection` as the rate on a `TransitionFlow`, with mixing from `MixingMatrix`.

| Flow | Source compartment? | Destination compartment? |
|---|---|---|
| `TransitionFlow` | Yes | Yes |
| Infection (`TransitionFlow` + `ForceOfInfection`) | Yes | Yes |
| `ExitFlow` (e.g. death) | Yes | No |
| `EntryFlow` (e.g. importation / birth) | No | Yes |

Flow rates need not be fixed constants: time-varying expressions and quantities
derived from the state can drive them, and rates can be adjusted under
stratification. Those topics appear in later chapters and in the user guide.
